In [1]:
!pip install python-dotenv kaggle


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
KAGGLE_USERNAME = os.getenv("KAGGLE_USERNAME")
KAGGLE_KEY = os.getenv("KAGGLE_KEY")

In [3]:
import os

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print("Kaggle authenticated successfully.")

Kaggle authenticated successfully.


In [4]:
DOWNLOAD_DIR = "../data/raw"

api.dataset_download_files(
    "aealemran/thesis-dataset2",
    path=DOWNLOAD_DIR,
    unzip=True
)

print("Dataset downloaded successfully.")

Dataset URL: https://www.kaggle.com/datasets/aealemran/thesis-dataset2


Dataset downloaded successfully.


In [5]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

for item in RAW_DIR.iterdir():
    print(item)

../data/raw/Thesis_dataset2


In [6]:
for path in RAW_DIR.rglob("*"):
    print(path)

../data/raw/Thesis_dataset2
../data/raw/Thesis_dataset2/DangerousDriving
../data/raw/Thesis_dataset2/Yawn
../data/raw/Thesis_dataset2/Openeye
../data/raw/Thesis_dataset2/Drinking
../data/raw/Thesis_dataset2/SafeDriving
../data/raw/Thesis_dataset2/Distracted
../data/raw/Thesis_dataset2/closed
../data/raw/Thesis_dataset2/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-868_jpg.rf.b9f2f685d1add96abf4bdfd4c048f340.jpg
../data/raw/Thesis_dataset2/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-527_jpg.rf.07cce399daf961f7053fb974622427c5.jpg
../data/raw/Thesis_dataset2/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-536_jpg.rf.fb126d91ccf81229fed49d2cf109c04d.jpg
../data/raw/Thesis_dataset2/DangerousDriving/gB_10_s2_2019-03-11T15-15-21-01-00_ir_face_mp4-559_jpg.rf.fabf4f552614f877deb084bd4cec9021.jpg
../data/raw/Thesis_dataset2/DangerousDriving/gB_10_s2_2019-03-11T15-15-21-01-00_ir_face_mp4-905_jpg.rf.2a2a6eeab3d1f53c068fb3dcbe5f2e05.jpg
../data/ra

____________________________ Discovering different datasets__________________________

In [9]:
from pathlib import Path

RAW_DATASET = Path("../data/raw/Thesis_dataset2")

for folder in RAW_DATASET.iterdir():
    if folder.is_dir():
        print("\n", folder.name)
        for f in list(folder.rglob("*"))[:10]:
            print("   ", f)


 DangerousDriving
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-868_jpg.rf.b9f2f685d1add96abf4bdfd4c048f340.jpg
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-527_jpg.rf.07cce399daf961f7053fb974622427c5.jpg
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-536_jpg.rf.fb126d91ccf81229fed49d2cf109c04d.jpg
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_10_s2_2019-03-11T15-15-21-01-00_ir_face_mp4-559_jpg.rf.fabf4f552614f877deb084bd4cec9021.jpg
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_10_s2_2019-03-11T15-15-21-01-00_ir_face_mp4-905_jpg.rf.2a2a6eeab3d1f53c068fb3dcbe5f2e05.jpg
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_6_s2_2019-03-11T13-46-14-01-00_ir_face_mp4-561_jpg.rf.db2381e40edba8884707338621f693a2.jpg
    ../data/raw/Thesis_dataset2/DangerousDriving/gB_10_s2_2019-03-11T15-15-21-01-00_ir_face_mp4-670_jpg.rf.08cbbd1fe1b89b

In [11]:
from pathlib import Path
import pandas as pd

IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

summary = []

for cls in classes:

    folder = RAW_DATASET / cls

    images = [
        f for f in folder.rglob("*")
        if f.suffix.lower() in IMAGE_EXTS
    ]

    summary.append({
        "class": cls,
        "images": len(images)
    })

summary = pd.DataFrame(summary)

summary

,class,images
0,DangerousDriving,2478
1,Distracted,1672
2,Drinking,2099
3,Openeye,2000
4,SafeDriving,2203
5,Yawn,2731
6,closed,1233


In [13]:
from pathlib import Path
import shutil
import random
import pandas as pd

random.seed(42)

# ----------------------------
# Paths
# ----------------------------
RAW_DATASET = Path("../data/raw/Thesis_dataset2")

SAMPLE_DIR = Path("../data/samples")

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Parameters
# ----------------------------
SAMPLES_PER_CLASS = 200   # Change to 300 or 500 later if required
RANDOM_SEED = 42

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

metadata = []
sampling_summary = []

# ----------------------------
# Sampling
# ----------------------------
for cls in sorted(classes):

    source_folder = RAW_DATASET / cls
    target_folder = SAMPLE_DIR / cls

    target_folder.mkdir(parents=True, exist_ok=True)

    images = [
        img for img in source_folder.rglob("*")
        if img.suffix.lower() in IMAGE_EXTENSIONS
    ]

    original_count = len(images)

    sample_size = min(SAMPLES_PER_CLASS, original_count)

    sampled_images = random.sample(images, sample_size)

    for img in sampled_images:

        shutil.copy2(
            img,
            target_folder / img.name
        )

        metadata.append({

            "file_name": img.name,
            "label": cls,
            "dataset": "Thesis_dataset2",
            "original_path": str(img)

        })

    sampling_summary.append({

        "class": cls,
        "original_images": original_count,
        "sampled_images": sample_size,
        "sampling_percentage": round(
            sample_size / original_count * 100,
            2
        )

    })

# ----------------------------
# Save Metadata
# ----------------------------
metadata = pd.DataFrame(metadata)

sampling_summary = pd.DataFrame(sampling_summary)

metadata.to_csv(
    SAMPLE_DIR / "metadata.csv",
    index=False
)

sampling_summary.to_csv(
    SAMPLE_DIR / "sampling_summary.csv",
    index=False
)

print("✅ Sampling Completed")
print(f"Total Sampled Images : {len(metadata)}")

display(sampling_summary)

✅ Sampling Completed
Total Sampled Images : 1400


,class,original_images,sampled_images,sampling_percentage
0,DangerousDriving,2478,200,8.07
1,Distracted,1672,200,11.96
2,Drinking,2099,200,9.53
3,Openeye,2000,200,10.00
4,SafeDriving,2203,200,9.08
5,Yawn,2731,200,7.32
6,closed,1233,200,16.22
